# MSI5001 SER

Compare seven untuned models on AESDD: DummyClassifier, KNN, Logistic Regression, 
Random Forest, SVM, MLP and CNN. The six scikit-learn classifiers retain their
default constructor settings (including MLP's 200-iteration limit).
CNN uses two convolution layers, Adam defaults, batch size 32 and 10 epochs.

Run sections 1–10 in order. Sections 1–8 establish the untuned baseline;
section 9 tunes each model using training-data validation, and section 10 compares
baseline and optimised predictions on the same outer test recordings. Set `FEATURE_SET` in section 4 to `'rich'`
for 120 acoustic features and log-mel + delta + delta-delta CNN inputs (default),
or `'basic'` for 40 MFCC mean/std features and log-mel-only CNN inputs.
Each CNN channel has 64 mel bands × 128 time bins. Evaluation uses five-fold stratified group random
CV and LOSO by speaker (six folds for AESDD). Alternate takes stay together in
random CV. Each recording is tested once per protocol. Report pooled out-of-fold
accuracy/macro F1 and fold mean ± standard deviation for both protocols.
Baseline and optimised models use identical outer folds.

## 1. Install dependency & import library

Installation is optional when the selected kernel already has the dependencies.
Run installation before imports and restart the kernel after upgrades if required.
For local NVIDIA GPU support, install a CUDA-enabled PyTorch build using the
[official selector](https://pytorch.org/get-started/locally/) in your environment;
skip the installation cell once that environment is ready.

In [ ]:
%pip install --upgrade numpy pandas scikit-learn librosa soundfile gdown matplotlib ipython torch

In [ ]:
from pathlib import Path
import re, zipfile, hashlib
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display, Audio, Markdown
import librosa
import soundfile as sf
import gdown
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold, LeaveOneGroupOut
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

## 2. Load dataset

Choose `local` or `drive`. Local input accepts a WAV directory, ZIP, or folder of ZIPs.
Drive input copies a public folder or dataset ZIP locally without mounting Drive.
The Drive sharing setting must be **Anyone with the link → Viewer**.
This section lists candidate AESDD files; audio validation and cleanup happen in section 3.

In [ ]:
SOURCE = 'local'                         # 'local' or 'drive'
LOCAL_PATH = r'./AESDD'                   # WAV directory, ZIP, or folder of ZIPs
DRIVE_URL = 'PASTE_PUBLIC_FILE_OR_FOLDER_URL_HERE'

root = Path(LOCAL_PATH).expanduser().resolve()
if SOURCE == 'drive':
    root = Path('./downloaded_AESDD').resolve()
    root.mkdir(exist_ok=True)
    if '/folders/' in DRIVE_URL:
        downloaded = gdown.download_folder(DRIVE_URL, output=str(root), use_cookies=False)
    else:
        downloaded = gdown.download(DRIVE_URL, output=str(root/'dataset.zip'), use_cookies=False)
    if not downloaded:
        raise RuntimeError('Download failed. Use a public anyone-with-the-link Drive URL.')
elif SOURCE != 'local':
    raise ValueError("SOURCE must be 'local' or 'drive'.")
if not root.exists():
    raise FileNotFoundError(f'Edit LOCAL_PATH: {root}')

def unzip(path):
    destination = path.with_suffix('')
    if not (destination/'.extracted').exists():
        with zipfile.ZipFile(path) as archive:
            if any(not (destination/name).resolve().is_relative_to(destination.resolve())
                   for name in archive.namelist()):
                raise ValueError('Unsafe archive path')
            archive.extractall(destination)
        (destination/'.extracted').touch()
    return destination

if root.is_file():
    root = unzip(root)
for _ in range(3):                       # Also handles ZIPs inside a downloaded folder
    for archive in list(root.rglob('*.zip')):
        unzip(archive)

codes = {'a':'Anger', 'd':'Disgust', 'f':'Fear', 'h':'Happiness', 's':'Sadness'}
pattern = re.compile(r'([adfhs])\s*[_ -]?(\d{1,2})\s*[_ -]?\(\s*(\d+)\s*\)\s*[a-z]*', re.I)
records = []
for path in sorted(root.rglob('*.wav')):
    match = pattern.fullmatch(path.stem)
    if not match or '__MACOSX' in path.parts:
        continue
    emotion, utterance, speaker = match.groups()
    emotion, speaker = emotion.lower(), str(int(speaker))
    records.append({'path':path, 'emotion':codes[emotion], 'speaker':speaker,
                    'group':f'{speaker}_{emotion}_{int(utterance)}'})
if not records:
    raise ValueError('No AESDD WAV filenames found. Check the dataset path and filenames.')
recordings = pd.DataFrame(records)
display(pd.DataFrame([{'Source':SOURCE, 'Dataset folder':str(root),
                       'Candidate recordings':len(recordings)}]))

## 3. Explore and clean dataset

Inspect the candidate counts, then remove byte-identical duplicates and skip
damaged, empty, non-finite or silent WAVs. Silence is checked on the mono signal;
the original samples are cached for preprocessing and playback.
Tables show counts before and after cleanup, removal reasons, speaker/emotion
distribution, and the first five usable recordings. Audio players preview those
five recordings. Playback normalization does not alter the model inputs.

In [ ]:
before_counts = recordings.emotion.value_counts().sort_index()
candidate_count = len(recordings)
clean_records, cleanup_log, audio_cache, seen = [], [], {}, set()
for record in recordings.to_dict('records'):
    path = record['path']
    try:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest in seen:
            cleanup_log.append({'File':path.name, 'Status':'Duplicate',
                                'Reason':'Byte-identical to a retained recording'})
            continue
        audio, sr = sf.read(path, dtype='float32', always_2d=True)
        if not len(audio) or sr <= 0:
            raise ValueError('Empty recording or invalid sample rate')
        if not np.isfinite(audio).all():
            raise ValueError('Non-finite audio samples')
        if np.max(np.abs(audio.mean(axis=1))) < 1e-7:
            raise ValueError('Silent mono signal')
    except (RuntimeError, ValueError, OSError) as error:
        cleanup_log.append({'File':path.name, 'Status':'Invalid', 'Reason':str(error)})
        continue
    clean_records.append({**record, 'duration_s':len(audio)/sr,
                          'sample_rate':sr, 'channels':audio.shape[1]})
    audio_cache[path] = (audio, sr)
    seen.add(digest)
if not clean_records:
    raise ValueError('No usable recordings remain after cleanup.')
recordings = pd.DataFrame(clean_records)
cleanup_log = pd.DataFrame(cleanup_log, columns=['File', 'Status', 'Reason'])
cleanup_summary = pd.DataFrame({
    'Recordings':[candidate_count, (cleanup_log.Status == 'Duplicate').sum(),
                  (cleanup_log.Status == 'Invalid').sum(), len(recordings)]
}, index=['Before cleanup', 'Removed duplicates', 'Skipped invalid WAVs', 'After cleanup'])
display(Markdown('**Cleanup summary**'))
display(cleanup_summary)
if not cleanup_log.empty:
    display(Markdown('**Removed recordings**'))
    display(cleanup_log)

emotion_counts = pd.DataFrame({'Before cleanup':before_counts,
                               'After cleanup':recordings.emotion.value_counts()}).fillna(0).astype(int)
emotion_counts = emotion_counts.sort_index()
speaker_counts = pd.crosstab(recordings.speaker, recordings.emotion)
display(Markdown('**Recordings per emotion**'))
display(emotion_counts)
display(Markdown('**Usable recordings per speaker and emotion**'))
display(speaker_counts)
display(Markdown('**Recording duration (seconds)**'))
display(recordings.duration_s.describe().drop('count').to_frame('Seconds').round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
emotion_counts.plot.bar(ax=axes[0], color=['#94a3b8', '#2563eb'], rot=25)
axes[0].set(title='Emotion balance before and after cleanup', xlabel='Emotion', ylabel='Recordings')
axes[0].set_ylim(0, emotion_counts.to_numpy().max()*1.2)
axes[0].legend(loc='upper center', ncol=2, frameon=False)
recordings.duration_s.plot.hist(ax=axes[1], bins=20, color='#0d9488', edgecolor='white')
axes[1].set(title='Duration of usable recordings', xlabel='Duration (seconds)', ylabel='Recordings')
fig.tight_layout()
plt.show()

ax = speaker_counts.plot.bar(stacked=True, figsize=(9, 4), colormap='Set2', rot=0)
ax.set(title='Emotion distribution by speaker after cleanup', xlabel='Speaker', ylabel='Recordings')
ax.legend(title='Emotion', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.figure.tight_layout()
plt.show()


In [ ]:
emotion_samples = (
    recordings.groupby('emotion')
    .sample(n=1)
    .sort_values('emotion')
    .copy()
)

emotion_samples['file'] = emotion_samples.path.map(lambda path: path.name)

display(Markdown('**One random recording per emotion**'))
display(
    emotion_samples[['file', 'emotion', 'speaker', 'duration_s', 'sample_rate', 'channels']]
    .reset_index(drop=True)
    .round({'duration_s': 2})
)

for i, record in enumerate(emotion_samples.itertuples(index=False), start=1):
    display(Markdown(
        f'**{i}. {record.path.name}** — {record.emotion}, speaker {record.speaker}'
    ))
    audio, sr = audio_cache[record.path]
    display(Audio(data=audio.T, rate=sr))

## 4. Data preprocessing

Convert to mono and resample to 16 kHz. One selector controls both model inputs:

| `FEATURE_SET` | Six scikit-learn models | CNN input per recording |
| --- | --- | --- |
| `'basic'` | 40 MFCC mean/std features | Log-mel only: **1 × 64 × 128** |
| `'rich'` (default) | 120 acoustic features | Log-mel + delta + delta-delta: **3 × 64 × 128** |

The rich acoustic features retain the following category order:

| Category | Feature | Summary statistics | Count |
| --- | --- | --- | ---: |
| Cepstral | MFCC | Mean/std of 20 coefficients | 40 |
| | Delta MFCC | Std of 20 first-order delta coefficients | 20 |
| Spectral | Centroid | Mean/std | 2 |
| | Bandwidth | Mean/std | 2 |
| | 85% roll-off | Mean/std | 2 |
| | Flatness | Mean/std | 2 |
| | Contrast | Mean/std across 7 frequency bands | 14 |
| Prosodic | Pitch (F0) | Mean, std, P5, P95; P5–P95 range in semitones | 5 |
| | Energy | Active-frame RMS dBFS mean, std, P5, P95 | 4 |
| | Timing and voicing | Pause ratio, pauses per second, voiced-frame ratio | 3 |
| Harmonic | Chroma | Mean/std across 12 pitch classes | 24 |
| Time-domain | Zero-crossing rate | Mean/std | 2 |
| **Total** | | | **120** |

MFCCs describe the spectral envelope. First-order delta MFCCs use a 9-frame window
with nearest-edge padding; retain each coefficient's standard deviation.
Spectral contrast uses `n_bands=6`, returning 7 bands. Flatness describes noise-like
versus tonal spectral structure. These summaries include the whole recording.

Pitch uses pYIN over a fixed 50–600 Hz range. Only finite, voiced estimates
contribute; all five pitch statistics are zero when none are detected.
The semitone range is `12 * log2(P95 / P5)`.

Energy/timing use 25 ms RMS windows with a 10 ms hop. A low-energy frame is more
than 30 dB below that clip's maximum frame RMS level. Internal low-energy runs
lasting at least 150 ms count as pause proxies; leading/trailing silence is excluded.
Pause duration and count are divided by the full clip duration. Energy statistics
use active frames. Voiced ratio is the fraction of pYIN frames flagged as voiced.
RMS dBFS is a digital level, not calibrated physical loudness. Equal peak levels
do not imply equal RMS levels. These rules are fixed per recording.

Chroma uses `librosa.feature.chroma_stft` with 12 pitch classes and its default
per-frame normalisation and fixed A440 reference (`tuning=0.0`). This bypasses
automatic tuning estimation, which triggered a Numba cast warning. Each pitch class contributes its
mean and standard deviation across the whole recording. Chroma combines octaves;
F0 separately retains pitch height. No individual feature determines an emotion.

Compared with the previous 97-feature rich set, add 24 chroma summaries and remove
clip duration from model inputs: **97 + 24 - 1 = 120**. Duration remains in metadata
and pause calculations; it also depends on the length of the spoken content.
Jitter/shimmer and onset rate remain omitted, as do extra energy-range statistics.

CNN rich channels are log-mel, its first-order temporal derivative (delta), and
its second-order temporal derivative (delta-delta), in that order. Derivatives use
`librosa.feature.delta` with a 9-frame window and nearest-edge padding along time.
Calculate both on the original log-mel matrix **before** resizing all channels
together to 64×128. These are log-mel derivatives, separate from the delta MFCC
summary features used by the scikit-learn models. No MFCC sequence enters the CNN.

Re-run sections **4–10** after changing `FEATURE_SET` or feature definitions. Basic
mode skips the other acoustic feature families. Baseline and tuned models use the
same selected inputs. Learned scaling stays in sections 6 and 9; no dataset-wide
statistics are fitted during extraction.

In [ ]:
FEATURE_SET = 'rich'  # 'basic': 40 MFCC + log-mel; 'rich': 120 features + log-mel/delta/delta-delta
if FEATURE_SET not in {'basic', 'rich'}:
    raise ValueError("FEATURE_SET must be 'basic' or 'rich'.")
CNN_CHANNEL_NAMES = ['Log-mel'] if FEATURE_SET == 'basic' else ['Log-mel', 'Delta', 'Delta-delta']
CNN_INPUT_NAME = ' + '.join(CNN_CHANNEL_NAMES)
feature_names = [f'mfcc_{i:02d}_{stat}' for stat in ['mean', 'std'] for i in range(20)]
if FEATURE_SET == 'rich':
    # Cepstral, Spectral, Prosodic, Harmonic, Time-domain.
    feature_names += [f'delta_mfcc_{i:02d}_std' for i in range(20)]
    feature_names += [f'{name}_{stat}' for name in
                      ['spectral_centroid_hz', 'spectral_bandwidth_hz', 'spectral_rolloff_hz',
                       'spectral_flatness'] for stat in ['mean', 'std']]
    feature_names += [f'spectral_contrast_band_{band}_{stat}' for band in range(7)
                      for stat in ['mean', 'std']]
    feature_names += [f'pitch_hz_{stat}' for stat in ['mean', 'std', 'p05', 'p95']]
    feature_names += ['pitch_range_semitones']
    feature_names += [f'rms_dbfs_{stat}' for stat in ['mean', 'std', 'p05', 'p95']]
    feature_names += ['pause_ratio', 'pauses_per_second', 'voiced_ratio']
    feature_names += [f'chroma_{pitch_class}_{stat}' for pitch_class in
                      ['C', 'C_sharp', 'D', 'D_sharp', 'E', 'F', 'F_sharp', 'G', 'G_sharp', 'A', 'A_sharp', 'B']
                      for stat in ['mean', 'std']]
    feature_names += ['zcr_mean', 'zcr_std']

def extract_ml_features(audio, feature_set=FEATURE_SET):
    mfcc = librosa.feature.mfcc(y=audio, sr=16000, n_mfcc=20)
    values = np.r_[mfcc.mean(axis=1), mfcc.std(axis=1)]
    if feature_set == 'basic':
        return values
    f0, voiced, _ = librosa.pyin(audio, fmin=50, fmax=600, sr=16000)
    pitch = f0[voiced & np.isfinite(f0)]
    pitch_stats = np.zeros(5)
    if pitch.size:
        p05, p95 = np.percentile(pitch, [5, 95])
        pitch_stats = [pitch.mean(), pitch.std(), p05, p95, 12*np.log2(p95/p05)]

    # Active-frame RMS statistics and explicitly defined low-energy pause proxies.
    rms = librosa.feature.rms(y=audio, frame_length=400, hop_length=160)[0]
    rms_db = 20*np.log10(np.maximum(rms, 1e-8))
    active = rms_db >= rms_db.max()-30
    levels = rms_db[active]
    energy_stats = np.r_[levels.mean(), levels.std(), np.percentile(levels, [5,95])]
    internal_silence = ~active
    active_frames = np.flatnonzero(active)
    internal_silence[:active_frames[0]] = False
    internal_silence[active_frames[-1]+1:] = False
    edges = np.diff(np.r_[0, internal_silence.astype(int), 0])
    starts, ends = np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)
    duration = len(audio)/16000
    pause_lengths = np.minimum(ends*0.01, duration)-np.minimum(starts*0.01, duration)
    pause_lengths = pause_lengths[pause_lengths >= 0.15]

    spectral_tracks = [
        librosa.feature.spectral_centroid(y=audio, sr=16000)[0],
        librosa.feature.spectral_bandwidth(y=audio, sr=16000)[0],
        librosa.feature.spectral_rolloff(y=audio, sr=16000, roll_percent=0.85)[0]
    ]
    spectral_stats = np.array([(track.mean(), track.std()) for track in spectral_tracks]).ravel()
    timing = [pause_lengths.sum()/duration, len(pause_lengths)/duration, voiced.mean()]
    zcr = librosa.feature.zero_crossing_rate(audio)[0]
    flatness = librosa.feature.spectral_flatness(y=audio)[0]
    delta_mfcc = librosa.feature.delta(mfcc, width=9, order=1, mode='nearest')
    contrast = librosa.feature.spectral_contrast(y=audio, sr=16000, n_bands=6)
    contrast_stats = np.array([(band.mean(), band.std()) for band in contrast]).ravel()
    chroma = librosa.feature.chroma_stft(y=audio, sr=16000, n_chroma=12, tuning=0.0)
    chroma_stats = np.array([(track.mean(), track.std()) for track in chroma]).ravel()
    return np.r_[values, delta_mfcc.std(axis=1),
                 spectral_stats, flatness.mean(), flatness.std(), contrast_stats,
                 pitch_stats, energy_stats, timing, chroma_stats, zcr.mean(), zcr.std()]

def extract_cnn_features(audio, feature_set=FEATURE_SET):
    if feature_set not in {'basic', 'rich'}:
        raise ValueError("feature_set must be 'basic' or 'rich'.")
    mel = librosa.feature.melspectrogram(y=audio, sr=16000, n_mels=64)
    log_mel = librosa.power_to_db(mel)
    channels = [log_mel]
    if feature_set == 'rich':
        # Differentiate along original time frames, before resizing.
        channels += [librosa.feature.delta(log_mel, width=9, order=order,
                     axis=-1, mode='nearest') for order in [1, 2]]
    tensor = torch.as_tensor(np.stack(channels), dtype=torch.float32)[None]
    return nn.functional.interpolate(tensor, size=(64,128), mode='bilinear',
                                     align_corners=False)[0].numpy()

def cnn_normalisation(train_images):
    # One mean/std per channel, using training clips, mel bands and time bins.
    axes = (0, 2, 3)
    mean = train_images.mean(axis=axes, keepdims=True)
    std = np.maximum(train_images.std(axis=axes, keepdims=True), 1e-8)
    return mean, std

features, images = [], []
for record in recordings.to_dict('records'):
    audio, sr = audio_cache[record['path']]
    audio = librosa.resample(audio.mean(axis=1), orig_sr=sr, target_sr=16000)
    image = extract_cnn_features(audio)
    if not features:
        example_audio = audio
    features.append(extract_ml_features(audio))
    images.append(image)
metadata = recordings.copy()
X, Z = np.asarray(features), np.asarray(images)
if X.shape[1] != len(feature_names) or not np.isfinite(X).all():
    raise ValueError('Acoustic feature count mismatch or non-finite values found.')
if Z.shape[1:] != (len(CNN_CHANNEL_NAMES),64,128) or not np.isfinite(Z).all():
    raise ValueError('CNN input shape mismatch or non-finite values found.')
encoder = LabelEncoder()
y = encoder.fit_transform(metadata.emotion)
display(pd.DataFrame([
    {'Input':f'Acoustic features ({FEATURE_SET})', 'Shape':str(X.shape), 'Used by':'Six scikit-learn models'},
    {'Input':CNN_INPUT_NAME, 'Shape':str(Z.shape), 'Used by':'CNN'},
    {'Input':'Emotion labels', 'Shape':str(y.shape), 'Used by':'All seven models'}
]))
display(pd.DataFrame({'Label':np.arange(len(encoder.classes_)), 'Emotion':encoder.classes_}))

feature_families = [('Cepstral', 'MFCC', 'Mean/std of 20 coefficients', 40)]
if FEATURE_SET == 'rich':
    feature_families += [
        ('Cepstral', 'Delta MFCC', 'Std of 20 first-order delta coefficients', 20),
        ('Spectral', 'Spectral centroid', 'Mean/std', 2),
        ('Spectral', 'Spectral bandwidth', 'Mean/std', 2),
        ('Spectral', 'Spectral roll-off', 'Mean/std; 85% energy threshold', 2),
        ('Spectral', 'Spectral flatness', 'Mean/std', 2),
        ('Spectral', 'Spectral contrast', 'Mean/std of 7 bands', 14),
        ('Prosodic', 'Pitch', 'Mean/std/P5/P95; semitone range', 5),
        ('Prosodic', 'Energy', 'Active-frame RMS dBFS mean/std/P5/P95', 4),
        ('Prosodic', 'Timing and voicing', 'Pause ratio/pause rate/voiced ratio', 3),
        ('Harmonic', 'Chroma', 'Mean/std of 12 pitch classes', 24),
        ('Time-domain', 'Zero-crossing rate', 'Mean/std', 2)
    ]
feature_table = pd.DataFrame(feature_families,
    columns=['Category', 'Feature', 'Statistics', 'Feature count'])
display(feature_table)
display(feature_table.groupby('Category', sort=False, as_index=False)['Feature count'].sum())

In [ ]:
example = metadata.iloc[0]
fig, axes = plt.subplots(1, 1+len(CNN_CHANNEL_NAMES), figsize=(4*(1+len(CNN_CHANNEL_NAMES)),3.5))
axes[0].plot(np.arange(len(example_audio))/16000, example_audio, color='#2563eb', linewidth=0.7)
axes[0].set(title=f'Mono waveform: {example.path.name}', xlabel='Time (seconds)', ylabel='Amplitude')
units = ['Power (dB)', 'Delta (dB/frame)', 'Delta-delta (dB/frame²)']
for channel, name in enumerate(CNN_CHANNEL_NAMES):
    image_plot = axes[channel+1].imshow(Z[0,channel], origin='lower', aspect='auto', cmap='magma')
    axes[channel+1].set(title=f'CNN input: {name}', xlabel='Resized time bin', ylabel='Mel band')
    fig.colorbar(image_plot, ax=axes[channel+1], label=units[channel])
fig.tight_layout()
plt.show()


## 5. Train test split (random & LOSO)

Random: use **all five folds** of a shuffled stratified group split. Each fold
trains on approximately 80% and tests on 20%. Alternate takes of the same
speaker/emotion/utterance stay together; speakers can appear on both sides.

LOSO: hold out one complete speaker at a time (**six folds** for AESDD). LOSO
follows speaker boundaries; it does not move recordings to enforce stratification.

Every recording appears in exactly one outer test fold per protocol. The same
outer folds are reused for baseline and optimised models. With AESDD, this creates
**11 outer training/test splits**. Section 9 adds inner validation within
each outer training split; it does not use the corresponding outer test data.

In [ ]:
OUTER_RANDOM_FOLDS = 5
random_cv = StratifiedGroupKFold(n_splits=OUTER_RANDOM_FOLDS, shuffle=True, random_state=42)
splits = [('Random', train, test) for train, test in random_cv.split(X, y, groups=metadata.group)]
splits += [('LOSO', train, test)
           for train, test in LeaveOneGroupOut().split(X, y, groups=metadata.speaker)]

# Both protocols must test each recording once; grouping rules must hold in every fold.
for setting in ['Random','LOSO']:
    protocol_splits = [(train,test) for protocol,train,test in splits if protocol==setting]
    tested = np.concatenate([test for _,test in protocol_splits])
    np.testing.assert_array_equal(np.sort(tested),np.arange(len(y)))
    group_column = 'group' if setting=='Random' else 'speaker'
    for train,test in protocol_splits:
        assert not set(train) & set(test)
        assert not set(metadata[group_column].iloc[train]) & set(metadata[group_column].iloc[test])

split_summary = pd.DataFrame([
    {'Split':(f'Random: fold {i+1}' if setting=='Random'
              else 'LOSO: speaker '+metadata.speaker.iloc[test].iloc[0]),
     'Protocol':setting, 'Train':len(train), 'Test':len(test),
     'Test speakers':','.join(metadata.speaker.iloc[test].unique())}
    for i,(setting,train,test) in enumerate(splits)
])
split_summary.index = pd.RangeIndex(1,len(split_summary)+1,name='Fold')
display(split_summary)
random_class_counts = pd.concat({
    split_name:pd.DataFrame({
        'Train':np.bincount(y[train],minlength=len(encoder.classes_)),
        'Test':np.bincount(y[test],minlength=len(encoder.classes_))
    },index=pd.Index(encoder.classes_,name='Emotion'))
    for split_name,(setting,train,test) in zip(split_summary.Split,splits) if setting=='Random'
},names=['Fold','Emotion'])
display(Markdown('**Emotion counts in all five random outer folds**'))
display(random_class_counts)

## 6. Feature scaling

Fit a fresh StandardScaler on each fold's **training acoustic features** and use it to
transform that fold's training and test features. Normalize CNN images using one
mean and standard deviation **per input channel**, computed from that fold's
**training images** across recordings, mel bands and time bins.
Test recordings never contribute to fitted scaling statistics.

The table has one row per independently scaled split: five random CV folds plus
one LOSO fold per speaker (11 rows for AESDD). Training-image mean/std should be approximately 0/1
after normalization in each nonconstant channel; the table summarises all channels.
Test data is transformed using those same training statistics.

In [ ]:
scaled_splits = []
for split_name, (setting, train, test) in zip(split_summary.Split, splits):
    scaler = StandardScaler().fit(X[train])
    mean, std = cnn_normalisation(Z[train])
    scaled_splits.append({
        'name':split_name, 'setting':setting, 'train':train, 'test':test,
        'X_train':scaler.transform(X[train]), 'X_test':scaler.transform(X[test]),
        'Z_train':(Z[train]-mean)/std, 'Z_test':(Z[test]-mean)/std,
        'y_train':y[train], 'y_test':y[test]
    })
display(pd.DataFrame([
    {'Split':fold['name'], 'Train clips':len(fold['train']), 'Test clips':len(fold['test']),
     'Acoustic features':fold['X_train'].shape[1], 'CNN channels':fold['Z_train'].shape[1],
     'CNN train mean':float(fold['Z_train'].mean()),
     'CNN train std':float(fold['Z_train'].std())}
    for fold in scaled_splits
]).set_index('Split').round(4))

## 7. Model training

Train fresh instances of all seven models for every split. Only training arrays
are passed into fitting. CNN uses CUDA automatically when available; the six
scikit-learn models use CPU. No hyperparameter tuning or early stopping is used.
Convergence warnings remain visible for models using their default iteration limit.
The first CNN convolution automatically accepts one channel in basic mode or three
in rich mode. All other baseline training settings remain the same.


In [ ]:
EPOCHS, BATCH_SIZE = 10, 32
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
display(pd.DataFrame([{
    'CNN device':str(DEVICE), 'GPU':torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'Not used',
    'PyTorch':torch.__version__, 'Epochs':EPOCHS, 'Batch size':BATCH_SIZE,
    'Feature set':FEATURE_SET, 'CNN input':CNN_INPUT_NAME, 'Input channels':Z.shape[1]
}]))
models = {'DummyClassifier':DummyClassifier(), 'KNN':KNeighborsClassifier(), 'Logistic Regression':LogisticRegression(),
          'Random Forest':RandomForestClassifier(), 'SVM':SVC(), 'MLP':MLPClassifier()}

def cnn_predict(model, images):
    model.eval()
    with torch.no_grad():
        return np.concatenate([model(torch.as_tensor(images[i:i+64], dtype=torch.float32).to(DEVICE))
            .argmax(1).cpu().numpy() for i in range(0,len(images),64)])

def train_cnn(images, labels):
    torch.manual_seed(42)
    inputs = torch.tensor(images)
    targets = torch.tensor(labels, dtype=torch.long)
    loader = DataLoader(TensorDataset(inputs, targets), batch_size=BATCH_SIZE, shuffle=True)
    model = nn.Sequential(
        nn.Conv2d(images.shape[1],16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32,len(encoder.classes_))
    ).to(DEVICE)
    optimizer, loss_fn = torch.optim.Adam(model.parameters()), nn.CrossEntropyLoss()
    for _ in range(EPOCHS):
        model.train()
        for batch, labels in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(batch.to(DEVICE)), labels.to(DEVICE))
            loss.backward()
            optimizer.step()
    return model

trained_splits = []
for fold in scaled_splits:
    fitted = {}
    for name, estimator in models.items():
        np.random.seed(42)
        fitted[name] = clone(estimator).fit(fold['X_train'], fold['y_train'])
    fitted['CNN'] = train_cnn(fold['Z_train'], fold['y_train'])
    trained_splits.append(fitted)
display(pd.DataFrame([
    {'Split':fold['name'], 'Trained models':len(fitted), 'Status':'Complete'}
    for fold, fitted in zip(scaled_splits, trained_splits)
]))

## 8. Model prediction and evaluation

Predict after baseline training on all **five random outer test folds** and all
**six LOSO outer test folds**. Pool out-of-fold predictions separately for each
protocol, then calculate accuracy and macro F1 using all five emotion labels.
Each recording contributes exactly once to each pooled score.

Also calculate each fold's metrics, then show their **unweighted mean ± sample
standard deviation (`ddof=1`)**: five values for random CV and six for LOSO.
Pooled macro F1 and mean fold macro F1 are different summaries; both are labelled.
Fold standard deviations describe variation across folds, not confidence intervals.
The charts show pooled scores. Accuracy uses percentages; macro F1 uses 0–1.

Training scores are additional diagnostics, not held-out performance estimates.
Show mean training/test accuracy and macro F1 across outer folds to help identify
underfitting or a large generalisation gap. These diagnostics do not change fitting.

In [ ]:
def summarise_predictions(records):
    """Return pooled OOF scores, individual fold scores, and fold mean/sample std."""
    fold_rows, pooled_rows = [], []
    class_labels = np.arange(len(encoder.classes_))
    for record in records:
        indices, actual, predicted = record['Indices'],record['Actual'],record['Predicted']
        if len(indices)!=len(actual) or len(actual)!=len(predicted):
            raise ValueError('Prediction lengths do not match the held-out recording indices.')
        np.testing.assert_array_equal(actual,y[indices])
        fold_rows.append({'Model':record['Model'], 'Protocol':record['Protocol'],
            'Fold':record['Fold'], 'Test clips':len(actual),
            'Accuracy':accuracy_score(actual,predicted),
            'Macro F1':f1_score(actual,predicted,average='macro',labels=class_labels,zero_division=0)})
    for name in [*models,'CNN']:
        row = {'Model':name}
        for setting in ['Random','LOSO']:
            chosen = [record for record in records if record['Model']==name and record['Protocol']==setting]
            if not chosen:
                raise ValueError(f'Missing out-of-fold predictions: {name}, {setting}.')
            indices = np.concatenate([record['Indices'] for record in chosen])
            if not np.array_equal(np.sort(indices),np.arange(len(y))):
                raise ValueError(f'{name}, {setting}: every recording must be tested exactly once.')
            if len({record['Fold'] for record in chosen})!=len(chosen):
                raise ValueError(f'Duplicate fold predictions: {name}, {setting}.')
            actual = np.concatenate([record['Actual'] for record in chosen])
            predicted = np.concatenate([record['Predicted'] for record in chosen])
            row[f'{setting} accuracy'] = accuracy_score(actual,predicted)
            row[f'{setting} macro F1'] = f1_score(actual,predicted,average='macro',
                                                 labels=class_labels,zero_division=0)
        pooled_rows.append(row)
    pooled = pd.DataFrame(pooled_rows).set_index('Model')
    per_fold = pd.DataFrame(fold_rows)
    fold_stats = per_fold.groupby(['Model','Protocol'],sort=False).agg(
        Folds=('Accuracy','size'), Accuracy_mean=('Accuracy','mean'),
        Accuracy_std=('Accuracy','std'), Macro_F1_mean=('Macro F1','mean'),
        Macro_F1_std=('Macro F1','std'))
    fold_stats.columns = ['Folds','Accuracy mean','Accuracy std','Macro F1 mean','Macro F1 std']
    fold_stats = fold_stats.reindex(pd.MultiIndex.from_product(
        [[*models,'CNN'],['Random','LOSO']],names=['Model','Protocol']))
    return pooled,per_fold,fold_stats

def show_fold_statistics(table,caption):
    view = table[['Folds']].copy()
    view['Accuracy (mean ± std)'] = [f'{mean:.1%} ± {std:.1%}'
                                    for mean,std in zip(table['Accuracy mean'],table['Accuracy std'])]
    view['Macro F1 (mean ± std)'] = [f'{mean:.3f} ± {std:.3f}'
                                    for mean,std in zip(table['Macro F1 mean'],table['Macro F1 std'])]
    display(view.style.set_caption(caption+' — sample std, ddof=1'))

if len(trained_splits)!=len(scaled_splits):
    raise RuntimeError('Re-run section 7 to train models for every current outer fold.')
predictions, prediction_records, baseline_training_rows = [], [], []
for fold,fitted in zip(scaled_splits,trained_splits):
    for name,model in fitted.items():
        if name=='CNN':
            predicted = cnn_predict(model, fold['Z_test'])
            training_predicted = cnn_predict(model, fold['Z_train'])
        else:
            predicted = model.predict(fold['X_test'])
            training_predicted = model.predict(fold['X_train'])
        baseline_training_rows.append({'Model':name, 'Protocol':fold['setting'], 'Fold':fold['name'],
            'Training accuracy':accuracy_score(fold['y_train'],training_predicted),
            'Training macro F1':f1_score(fold['y_train'],training_predicted,average='macro',
                                       labels=np.arange(len(encoder.classes_)),zero_division=0),
            'Test accuracy':accuracy_score(fold['y_test'],predicted),
            'Test macro F1':f1_score(fold['y_test'],predicted,average='macro',
                                   labels=np.arange(len(encoder.classes_)),zero_division=0)})
        predictions.append((fold['setting'],name,fold['y_test'].copy(),predicted.copy()))
        prediction_records.append({'Protocol':fold['setting'],'Fold':fold['name'],'Model':name,
            'Indices':fold['test'].copy(),'Actual':fold['y_test'].copy(),'Predicted':predicted.copy()})
summary,fold_scores,fold_summary = summarise_predictions(prediction_records)
display(summary.style.format({
    'Random accuracy':'{:.1%}','Random macro F1':'{:.3f}',
    'LOSO accuracy':'{:.1%}','LOSO macro F1':'{:.3f}'
}).set_caption(f'Pooled out-of-fold baseline — {FEATURE_SET} ({X.shape[1]} features); random 5-fold CV and LOSO; CNN {CNN_INPUT_NAME}'))
show_fold_statistics(fold_summary,'Baseline fold mean and standard deviation')
baseline_training_scores = pd.DataFrame(baseline_training_rows)
baseline_training_summary = baseline_training_scores.groupby(['Model','Protocol'],sort=False)[
    ['Training accuracy','Training macro F1','Test accuracy','Test macro F1']].mean()
display(Markdown('**Baseline diagnostics: mean training/test scores across outer folds**'))
display(baseline_training_summary.style.format({
    'Training accuracy':'{:.1%}', 'Test accuracy':'{:.1%}',
    'Training macro F1':'{:.3f}', 'Test macro F1':'{:.3f}'}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, metric, columns in zip(axes, ['Accuracy', 'Macro F1'],
        [['Random accuracy', 'LOSO accuracy'], ['Random macro F1', 'LOSO macro F1']]):
    values = summary[columns].copy()
    values.columns = ['Random CV (pooled)', 'LOSO (pooled)']
    values.plot.barh(ax=ax, color=['#2563eb', '#f59e0b'])
    ax.set(title=metric, xlabel=metric, ylabel='', xlim=(0, 1))
    ax.grid(axis='x', alpha=0.2)
    ax.set_axisbelow(True)
    ax.legend(loc='lower right')
axes[0].invert_yaxis()
axes[0].xaxis.set_major_formatter(PercentFormatter(1))
fig.suptitle('SER baseline: pooled out-of-fold random CV vs LOSO')
fig.tight_layout()
plt.show()

## 9. Model Optimisation & Hyperparameter Tuning

Keep the input preset, all its features and the outer splits fixed. Tune model
hyperparameters without selecting feature subsets. Each subsection
shows its search space, validation results and selected settings. DummyClassifier
remains the reference; it is not tuned. Outer-test scores are calculated only in
section 10 for the optimised models, after all selections have been made.

### 9.1 Baseline results

Preserve pooled scores, fold statistics and indexed predictions from section 8
for the selected feature set. Both stages reuse all five random and six LOSO folds.
This is a snapshot, not a second baseline training run. To compare basic and rich
feature modes, run sections 4–10 separately for each choice and record both runs.

In [ ]:
baseline_summary = summary.copy(deep=True)
baseline_predictions = [(setting, name, actual.copy(), predicted.copy())
                        for setting, name, actual, predicted in predictions]
baseline_prediction_records = [{key:(value.copy() if isinstance(value,np.ndarray) else value)
                                for key,value in record.items()} for record in prediction_records]
baseline_fold_scores = fold_scores.copy(deep=True)
baseline_fold_summary = fold_summary.copy(deep=True)
baseline_feature_set = FEATURE_SET
baseline_feature_count = X.shape[1]
baseline_cnn_shape = tuple(Z.shape[1:])
baseline_cnn_input = CNN_INPUT_NAME
model_order = [*models, 'CNN']
metric_format = {'Random accuracy':'{:.1%}', 'Random macro F1':'{:.3f}',
                 'LOSO accuracy':'{:.1%}', 'LOSO macro F1':'{:.3f}'}
display(baseline_summary.style.format(metric_format).set_caption(
    f'Pooled out-of-fold baseline — {baseline_feature_set}, {baseline_feature_count} features; CNN {baseline_cnn_input}'))
show_fold_statistics(baseline_fold_summary,'Baseline fold mean and standard deviation')
display(pd.DataFrame([
    {'Model':name, 'Baseline settings':repr(estimator)}
    for name, estimator in models.items()
] + [{'Model':'CNN', 'Baseline settings':f'{Z.shape[1]} input channels ({CNN_INPUT_NAME}), Conv 16/32, global average pooling, Adam lr=0.001, 10 epochs'}]))
display(Markdown('**Baseline training/test diagnostics — unweighted fold means**'))
display(baseline_training_summary.round(4))

### 9.2 Optimisation method

Tune separately inside every outer training split: five random folds and six
LOSO folds. Default inner validation uses **four stratified group random folds**
and **all remaining speakers** for LOSO (five inner holdouts for AESDD).
The outer test speaker never enters inner training or validation. A coverage
table shows how often each speaker serves as an inner validation holdout.

Classical models tune model settings using `Pipeline(StandardScaler, model)`.
Every candidate uses all features from the section 4 preset: 40 MFCC features
in basic mode or all 120 acoustic features in rich mode. This same fixed
representation is used for the baseline and every tuned model and outer fold.
The global basic/rich toggle continues to control both model families.

Every inner fold fits its own scaler, and every selected pipeline refits on all
outer training clips. CNN channel normalisation follows the same training-only
rule. Section 6's outer-scaled arrays are not used for inner tuning.

Select classical settings and CNN candidates by **mean inner-validation macro F1**;
retain accuracy as a second metric. For CNN, lowest inner-validation loss chooses
each checkpoint. Its macro F1 enters candidate selection. Median checkpoint
epoch and validation-derived learning-rate schedules determine a fresh outer
training refit. No outer test data select settings, checkpoints or epoch counts.

Tables display complete settings and training/validation scores. Each classical
section plots the selected candidates' mean inner-training versus validation
macro F1; these are diagnostics, not outer evaluation scores. Training F1 alone,
particularly for distance-weighted KNN, is not evidence of generalisation.

`TUNING_JOBS = 1` limits parallel searches and memory use; CNN automatically uses
CPU/CUDA. The richer searches and full speaker validation take substantially
longer than the previous compact run. Keep default baselines unchanged.
Re-running this cell clears tuning state, so re-run 9.3–10 afterwards.

In [ ]:
from time import perf_counter
import json
from sklearn.model_selection import GridSearchCV, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer

TUNING_SEED = 42
TUNING_JOBS = 1
INNER_RANDOM_FOLDS = 4
INNER_LOSO_FOLDS = None                 # All remaining speakers (five for AESDD)
CNN_MAX_EPOCHS = 100
CNN_MIN_EPOCHS = 20                     # Minimum before stopping, not a minimum checkpoint epoch
CNN_PATIENCE = 15
CNN_LR_PATIENCE = 5
CNN_MIN_LR = 1e-5

if (FEATURE_SET != baseline_feature_set or X.shape[1] != baseline_feature_count
        or tuple(Z.shape[1:]) != baseline_cnn_shape):
    raise RuntimeError('Feature preset changed: re-run sections 4–9.1 before tuning.')
labels = np.arange(len(encoder.classes_))
macro_f1_scorer = make_scorer(f1_score, average='macro', labels=labels, zero_division=0)

def format_settings(params):
    readable = {key.removeprefix('model__'):value for key, value in params.items()}
    return json.dumps(readable)

tuning_folds, validation_rows = [], []
for fold in scaled_splits:
    outer_train, outer_test = fold['train'], fold['test']
    subset = metadata.iloc[outer_train]
    if fold['setting'] == 'Random':
        inner = list(StratifiedGroupKFold(n_splits=INNER_RANDOM_FOLDS, shuffle=True,
                     random_state=TUNING_SEED).split(X[outer_train], y[outer_train], subset.group))
        group_column = 'group'
    else:
        inner = list(LeaveOneGroupOut().split(X[outer_train], y[outer_train], subset.speaker))
        if INNER_LOSO_FOLDS is not None:
            if not 1 <= INNER_LOSO_FOLDS <= len(inner):
                raise ValueError('INNER_LOSO_FOLDS must be None or a valid number of training speakers.')
            chosen = np.linspace(0, len(inner)-1, INNER_LOSO_FOLDS, dtype=int)
            inner = [inner[i] for i in chosen]
        group_column = 'speaker'
    for inner_id, (train_idx, valid_idx) in enumerate(inner, start=1):
        train_global, valid_global = outer_train[train_idx], outer_train[valid_idx]
        assert not set(train_global) & set(valid_global)
        assert not (set(train_global) | set(valid_global)) & set(outer_test)
        assert not set(subset[group_column].iloc[train_idx]) & set(subset[group_column].iloc[valid_idx])
        if len(np.unique(y[train_global])) != len(labels):
            raise ValueError('An inner training fold is missing an emotion; review the split design.')
        validation_rows.append({'Outer split':fold['name'], 'Protocol':fold['setting'], 'Inner fold':inner_id,
            'Train clips':len(train_idx), 'Validation clips':len(valid_idx),
            'Validation speakers':','.join(sorted(subset.speaker.iloc[valid_idx].unique())),
            'Group rule':group_column})
    tuning_folds.append({'name':fold['name'], 'setting':fold['setting'],
                         'train':outer_train.copy(), 'test':outer_test.copy(), 'inner':inner})
optimised_models = {fold['name']:{'DummyClassifier':trained_splits[i]['DummyClassifier']}
                   for i, fold in enumerate(tuning_folds)}
validation_results, selected_settings = {}, []
validation_plan = pd.DataFrame(validation_rows)
display(validation_plan)
display(Markdown('**Inner LOSO validation coverage across outer folds**'))
coverage = validation_plan.loc[validation_plan.Protocol == 'LOSO', 'Validation speakers'].str.split(',').explode()
display(coverage.value_counts().sort_index().rename_axis('Speaker').to_frame('Validation holdouts'))
display(pd.DataFrame([{'Selection metric':'Mean inner-validation macro F1',
    'Acoustic features':X.shape[1], 'Feature set':FEATURE_SET,
    'CNN input':CNN_INPUT_NAME, 'CNN channels':Z.shape[1],
    'Random outer folds':sum(fold['setting']=='Random' for fold in tuning_folds),
    'LOSO outer folds':sum(fold['setting']=='LOSO' for fold in tuning_folds),
    'Random inner folds':INNER_RANDOM_FOLDS, 'LOSO inner holdouts':INNER_LOSO_FOLDS or 'All',
    'CNN max epochs':CNN_MAX_EPOCHS, 'CNN patience':CNN_PATIENCE, 'CNN device':str(DEVICE)}]))

def tune_classical(name, estimator, param_grid):
    candidate_tables, selections = [], []
    selected_settings[:] = [row for row in selected_settings if row['Model'] != name]
    display(pd.DataFrame([{'Candidate':i+1, 'Settings':format_settings(params)}
                         for i, params in enumerate(ParameterGrid(param_grid))]))
    for fold in tuning_folds:
        display(Markdown(f"**{name}: {fold['name']}** — tuning on outer training data"))
        pipeline = Pipeline([('scale', StandardScaler()), ('model', clone(estimator))])
        search = GridSearchCV(pipeline, param_grid, cv=fold['inner'],
            scoring={'accuracy':'accuracy', 'macro_f1':macro_f1_scorer}, refit='macro_f1',
            n_jobs=TUNING_JOBS, error_score='raise', return_train_score=True)
        np.random.seed(TUNING_SEED)
        started = perf_counter()
        search.fit(X[fold['train']], y[fold['train']])
        elapsed = perf_counter()-started
        optimised_models[fold['name']][name] = search.best_estimator_
        result = search.cv_results_
        candidate_tables.append(pd.DataFrame({
            'Outer split':fold['name'], 'Protocol':fold['setting'],
            'Candidate':np.arange(1, len(result['params'])+1),
            'Settings':[format_settings(p) for p in result['params']],
            'Validation macro F1':result['mean_test_macro_f1'],
            'Validation F1 std':result['std_test_macro_f1'],
            'Validation accuracy':result['mean_test_accuracy'],
            'Training macro F1':result['mean_train_macro_f1'],
            'Training accuracy':result['mean_train_accuracy'],
            'Train-validation F1 gap':result['mean_train_macro_f1']-result['mean_test_macro_f1'],
            'Selected':np.arange(len(result['params'])) == search.best_index_
        }))
        selected = search.best_index_
        row = {'Model':name, 'Outer split':fold['name'], 'Protocol':fold['setting'], 'Feature set':FEATURE_SET,
            'Feature count':X.shape[1],
            'Selected settings':format_settings(search.best_params_), 'Validation macro F1':search.best_score_,
            'Validation accuracy':result['mean_test_accuracy'][selected],
            'Training macro F1':result['mean_train_macro_f1'][selected],
            'Train-validation F1 gap':result['mean_train_macro_f1'][selected]-search.best_score_,
            'Inner folds':len(fold['inner']), 'Search seconds':elapsed}
        selections.append(row)
        selected_settings.append(row)
    table = pd.concat(candidate_tables, ignore_index=True)
    validation_results[name] = table
    example_fold = next(fold['name'] for fold in tuning_folds if fold['setting']=='Random')
    display(Markdown(f'**Top 15 candidates: {example_fold}** — all candidates remain in `validation_results`'))
    display(table[table['Outer split']==example_fold].sort_values('Validation macro F1', ascending=False)
            .head(15).round(4))
    display(Markdown('**Selected configuration for each outer split**'))
    display(pd.DataFrame(selections).round(4))
    chosen = table[table.Selected]
    diagnostics = chosen.groupby('Protocol')[['Training macro F1', 'Validation macro F1',
                                             'Train-validation F1 gap']].mean().reindex(['Random','LOSO'])
    display(Markdown('**Selected candidates: mean inner-training and validation scores across outer folds**'))
    display(diagnostics.round(4))
    ax = diagnostics[['Training macro F1','Validation macro F1']].plot.bar(
        figsize=(7,3.5), color=['#94a3b8','#2563eb'], rot=0, ylim=(0,1))
    ax.set(title=f'{name}: inner-training versus validation', ylabel='Macro F1', xlabel='Protocol')
    ax.figure.tight_layout()
    plt.show()
    return table

### 9.3 KNN optimisation

Baseline: 5 neighbours, uniform weights and Euclidean distance. Compare
**3/5/7/9/11** neighbours, uniform/distance weighting and Manhattan/Euclidean
distance. This gives 20 model combinations, each using all features from the
selected basic/rich preset. Inner validation chooses each outer fold's settings.

In [ ]:
knn_grid = {'model__n_neighbors':[3,5,7,9,11], 'model__weights':['uniform','distance'],
            'model__p':[1,2]}
knn_validation = tune_classical('KNN', KNeighborsClassifier(), knn_grid)

### 9.4 Logistic Regression optimisation

Baseline: `C=1`, `max_iter=100`. Keep the default reference and compare
`C=0.1/1/10` with up to 1500 iterations, using all features from the selected preset.
Lower C means stronger regularisation. More iterations allow convergence;
they do not guarantee higher held-out scores when training already fits well.

In [ ]:
lr_grid = [
    {'model__C':[1.0], 'model__max_iter':[100]},
    {'model__C':[0.1,1.0,10.0], 'model__max_iter':[1500]}
]
lr_validation = tune_classical('Logistic Regression', LogisticRegression(), lr_grid)

### 9.5 Random Forest optimisation

Keep the default reference and previous structural candidates. Add two more
regularised candidates with larger leaves, bounded depth and `max_features=0.5`.
Six model configurations explore tree complexity using all features from the
selected preset; increasing tree count alone mainly improves
ensemble stability. All tuned forests use seed 42.

In [ ]:
rf_grid = [
    {'model__n_estimators':[100], 'model__max_depth':[None], 'model__min_samples_leaf':[1],
     'model__max_features':['sqrt']},
    {'model__n_estimators':[100], 'model__max_depth':[10], 'model__min_samples_leaf':[2],
     'model__max_features':['sqrt']},
    {'model__n_estimators':[100], 'model__max_depth':[None], 'model__min_samples_leaf':[4],
     'model__max_features':['sqrt']},
    {'model__n_estimators':[300], 'model__max_depth':[None], 'model__min_samples_leaf':[2],
     'model__max_features':['sqrt']},
    {'model__n_estimators':[100], 'model__max_depth':[6], 'model__min_samples_leaf':[8],
     'model__max_features':[0.5]},
    {'model__n_estimators':[100], 'model__max_depth':[10], 'model__min_samples_leaf':[4],
     'model__max_features':[0.5]}
]
rf_validation = tune_classical('Random Forest', RandomForestClassifier(random_state=TUNING_SEED), rf_grid)

### 9.6 SVM optimisation

Broaden the joint RBF search: `C=0.01/0.1/1/10/100/1000` and
`gamma='scale'/0.0001/0.001/0.01/0.1`. Retain linear C=1/10 comparisons.
There are 32 model combinations, each using all features from the selected preset.
C controls regularisation and gamma controls RBF locality; tune them jointly.
The default RBF reference is included.

In [ ]:
svm_grid = [
    {'model__kernel':['rbf'], 'model__C':[0.01,0.1,1,10,100,1000],
     'model__gamma':['scale',0.0001,0.001,0.01,0.1]},
    {'model__kernel':['linear'], 'model__C':[1,10]}
]
svm_validation = tune_classical('SVM', SVC(), svm_grid)

### 9.7 MLP optimisation

Preserve the default 100-neuron/200-iteration reference and its 1500-iteration
comparison. Also compare compact 32/64-neuron networks, Adam versus L-BFGS, and
`alpha=0.0001/0.01/0.1`: 14 model combinations, each using all features from the
selected preset. Stronger regularisation and smaller networks address the observed
training/validation gap. L-BFGS is an alternative worth testing on small datasets;
it does not use Adam's learning-rate parameter.

The untuned baseline remains at 200 iterations. Tuning allows up to 1500; warnings
remain visible if convergence is not reached. Internal random early stopping is
disabled so all validation respects our explicit grouping. The seed is fixed at 42.

In [ ]:
mlp_grid = [
    {'model__solver':['adam'], 'model__hidden_layer_sizes':[(100,)],
     'model__alpha':[0.0001], 'model__max_iter':[200]},
    {'model__solver':['adam'], 'model__hidden_layer_sizes':[(100,)],
     'model__alpha':[0.0001], 'model__max_iter':[1500]},
    {'model__solver':['adam'], 'model__hidden_layer_sizes':[(32,), (64,)],
     'model__alpha':[0.0001,0.01,0.1], 'model__max_iter':[1500]},
    {'model__solver':['lbfgs'], 'model__hidden_layer_sizes':[(32,), (64,)],
     'model__alpha':[0.0001,0.01,0.1], 'model__max_iter':[1500]}
]
mlp_validation = tune_classical('MLP', MLPClassifier(random_state=TUNING_SEED, early_stopping=False), mlp_grid)

### 9.8 CNN optimisation

Preserve the original two-convolution, 10-epoch CNN baseline in section 7.
Keep basic/rich CNN inputs fixed during tuning. Compare a compact spatial-pooling
reference with a stronger three-convolution network (16/32/64 channels), batch
normalisation, 4×4 spatial pooling, a 64-neuron dense layer and dropout. Test two
learning rates for the stronger architecture. All candidates use Adam and weight decay.

Each inner run can train for up to 100 epochs. Monitor **validation loss** for
checkpoints, learning-rate reduction and stopping. Reduce LR by half with scheduler
patience 5 and an absolute loss threshold of 0.0001, to a minimum of 1e-5. Stop after 15
non-improving epochs, once at least 20 epochs have run. The minimum delays stopping;
the best checkpoint can still occur before epoch 20, and so can the final refit epoch.

Restore a copied lowest-loss checkpoint inside each inner fold. Evaluate its macro
F1 and accuracy; mean checkpoint macro F1 selects the candidate. Obtain refit epochs
from the rounded-up median of its inner best epochs. Refit fresh weights on all
outer training clips, using the median inner learning rate at each refit epoch.
Extend shorter inner schedules with their last recorded rate. The outer test set
never controls training or LR scheduling; inner weights are not reused for refitting.

Tables include training/validation F1 gaps and selected epochs. Curves show loss,
macro F1 and LR for the selected candidate in the **first outer fold and first
inner fold of each protocol**, fixed independently of outer test performance.
Stronger architecture and longer training are hypotheses, not guaranteed gains;
the exploratory fixed 10/30-epoch results are not substituted for nested evaluation.

In [ ]:
cnn_candidates = [
    {'name':'Compact spatial reference', 'architecture':'compact',
     'learning_rate':0.001, 'dropout':0.3, 'weight_decay':0.0001},
    {'name':'Stronger CNN', 'architecture':'strong',
     'learning_rate':0.001, 'dropout':0.2, 'weight_decay':0.0001},
    {'name':'Stronger CNN, lower learning rate', 'architecture':'strong',
     'learning_rate':0.0003, 'dropout':0.2, 'weight_decay':0.0001}
]
display(pd.DataFrame(cnn_candidates))

def build_optimised_cnn(config, in_channels):
    if config['architecture'] == 'compact':
        layers = [
            nn.Conv2d(in_channels,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((2,2)), nn.Flatten(), nn.Dropout(config['dropout']),
            nn.Linear(32*2*2,len(encoder.classes_))]
    elif config['architecture'] == 'strong':
        layers = [
            nn.Conv2d(in_channels,16,3,padding=1,bias=False), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1,bias=False), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1,bias=False), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4,4)), nn.Flatten(), nn.Linear(64*4*4,64), nn.ReLU(),
            nn.Dropout(config['dropout']), nn.Linear(64,len(encoder.classes_))]
    else:
        raise ValueError('Unknown CNN architecture.')
    return nn.Sequential(*layers).to(DEVICE)

def cnn_epoch_metrics(model, images, targets):
    model.eval()
    predictions, loss_total = [], 0.0
    loss_fn = nn.CrossEntropyLoss(reduction='sum')
    with torch.no_grad():
        for i in range(0,len(images),64):
            logits = model(torch.as_tensor(images[i:i+64], dtype=torch.float32).to(DEVICE))
            target = torch.as_tensor(targets[i:i+64], dtype=torch.long).to(DEVICE)
            loss_total += float(loss_fn(logits, target))
            predictions.append(logits.argmax(1).cpu().numpy())
    predicted = np.concatenate(predictions)
    return {'loss':loss_total/len(targets), 'accuracy':accuracy_score(targets,predicted),
            'macro_f1':f1_score(targets,predicted,average='macro',labels=labels,zero_division=0)}

def fit_optimised_cnn(train_images, train_labels, config, epochs,
                      valid_images=None, valid_labels=None, seed=TUNING_SEED,
                      learning_rate_schedule=None):
    if epochs < 1 or CNN_PATIENCE < 1 or CNN_MIN_EPOCHS < 1:
        raise ValueError('CNN epochs, patience and minimum training epochs must be positive.')
    if (valid_images is None) != (valid_labels is None):
        raise ValueError('Supply both validation images and labels, or neither.')
    if learning_rate_schedule is not None and len(learning_rate_schedule) != epochs:
        raise ValueError('The validation-derived learning-rate schedule must match refit epochs.')
    torch.manual_seed(seed)
    model = build_optimised_cnn(config, train_images.shape[1])
    loader = DataLoader(TensorDataset(torch.as_tensor(train_images, dtype=torch.float32),
                       torch.as_tensor(train_labels, dtype=torch.long)), batch_size=BATCH_SIZE,
                       shuffle=True, generator=torch.Generator().manual_seed(seed))
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'],
                                 weight_decay=config['weight_decay'])
    scheduler = (torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=CNN_LR_PATIENCE,
        threshold=1e-4, threshold_mode='abs', min_lr=CNN_MIN_LR)
        if valid_images is not None else None)
    loss_fn = nn.CrossEntropyLoss()
    history, best_state = [], None
    best_epoch, best_loss, stale = 0, np.inf, 0
    for epoch in range(1, epochs+1):
        if learning_rate_schedule is not None:
            for group in optimizer.param_groups:
                group['lr'] = float(learning_rate_schedule[epoch-1])
        row = {'Epoch':epoch, 'Learning rate':optimizer.param_groups[0]['lr']}
        model.train()
        loss_total = 0.0
        for batch, target in loader:
            batch, target = batch.to(DEVICE), target.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(batch), target)
            loss.backward()
            optimizer.step()
            loss_total += float(loss.detach())*len(target)
        row['Training loss'] = row['Optimisation loss'] = loss_total/len(train_labels)
        if valid_images is not None:
            train_metrics = cnn_epoch_metrics(model, train_images, train_labels)
            valid_metrics = cnn_epoch_metrics(model, valid_images, valid_labels)
            row.update({'Training loss':train_metrics['loss'],
                        'Training macro F1':train_metrics['macro_f1'],
                        'Training accuracy':train_metrics['accuracy'],
                        'Validation loss':valid_metrics['loss'],
                        'Validation macro F1':valid_metrics['macro_f1'],
                        'Validation accuracy':valid_metrics['accuracy']})
            if valid_metrics['loss'] < best_loss-1e-4:
                best_epoch, best_loss, stale = epoch, valid_metrics['loss'], 0
                best_state = {key:value.detach().cpu().clone() for key,value in model.state_dict().items()}
            else:
                stale += 1
            scheduler.step(valid_metrics['loss'])
        history.append(row)
        if (valid_images is not None and epoch >= min(CNN_MIN_EPOCHS,epochs)
                and stale >= CNN_PATIENCE):
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    else:
        best_epoch = epochs
    history = pd.DataFrame(history)
    history['Selected checkpoint'] = ((history.Epoch == best_epoch)
                                      if valid_images is not None else False)
    return model, history, best_epoch

def refit_learning_rates(histories, epochs):
    # Some inner runs stop earlier. Extend those schedules with their last LR.
    return [float(np.median([history['Learning rate'].iloc[min(epoch,len(history)-1)]
                             for history in histories])) for epoch in range(epochs)]

selected_settings[:] = [row for row in selected_settings if row['Model'] != 'CNN']
cnn_history_parts, cnn_search_rows = [], []
for fold in tuning_folds:
    display(Markdown(f"**CNN: {fold['name']}** — inner validation then outer-training refit"))
    candidate_rows, candidate_histories = [], []
    for candidate_id, config in enumerate(cnn_candidates):
        scores, accuracies, training_scores, best_epochs, histories = [], [], [], [], []
        for inner_id, (train_idx, valid_idx) in enumerate(fold['inner'], start=1):
            train_global, valid_global = fold['train'][train_idx], fold['train'][valid_idx]
            image_mean, image_std = cnn_normalisation(Z[train_global])
            model, history, best_epoch = fit_optimised_cnn(
                (Z[train_global]-image_mean)/image_std, y[train_global], config, CNN_MAX_EPOCHS,
                (Z[valid_global]-image_mean)/image_std, y[valid_global], seed=TUNING_SEED+inner_id)
            best_row = history.loc[history.Epoch==best_epoch].iloc[0]
            scores.append(best_row['Validation macro F1'])
            accuracies.append(best_row['Validation accuracy'])
            training_scores.append(best_row['Training macro F1'])
            best_epochs.append(best_epoch)
            history['Outer split'], history['Protocol'] = fold['name'], fold['setting']
            history['Candidate'], history['Inner fold'] = config['name'], inner_id
            histories.append(history)
            cnn_history_parts.append(history)
            del model
        candidate_histories.append(histories)
        candidate_rows.append({'Outer split':fold['name'], 'Protocol':fold['setting'],
            'Candidate':config['name'], 'Candidate index':candidate_id,
            'Validation macro F1':float(np.mean(scores)), 'Validation F1 std':float(np.std(scores)),
            'Validation accuracy':float(np.mean(accuracies)), 'Training macro F1':float(np.mean(training_scores)),
            'Train-validation F1 gap':float(np.mean(training_scores)-np.mean(scores)),
            'Refit epochs':int(np.ceil(np.median(best_epochs))), 'Inner best epochs':str(best_epochs),
            'Checkpoint rule':'Lowest inner-validation loss', 'Settings':json.dumps(config)})
    best = max(candidate_rows, key=lambda row:row['Validation macro F1'])
    for row in candidate_rows:
        row['Selected'] = row['Candidate index']==best['Candidate index']
        cnn_search_rows.append(row)
    selected_config = cnn_candidates[best['Candidate index']]
    learning_rates = refit_learning_rates(candidate_histories[best['Candidate index']], best['Refit epochs'])
    image_mean, image_std = cnn_normalisation(Z[fold['train']])
    final_cnn, _, _ = fit_optimised_cnn((Z[fold['train']]-image_mean)/image_std,
                      y[fold['train']], selected_config, best['Refit epochs'],
                      learning_rate_schedule=learning_rates)
    optimised_models[fold['name']]['CNN'] = {'model':final_cnn.cpu(),
        'image_mean':image_mean, 'image_std':image_std, 'config':selected_config.copy(),
        'epochs':best['Refit epochs'], 'learning_rate_schedule':learning_rates}
    selected_settings.append({'Model':'CNN', 'Outer split':fold['name'], 'Protocol':fold['setting'],
        'Feature set':FEATURE_SET, 'CNN input':CNN_INPUT_NAME,
        'Selected settings':json.dumps(selected_config), 'Validation macro F1':best['Validation macro F1'],
        'Validation accuracy':best['Validation accuracy'], 'Training macro F1':best['Training macro F1'],
        'Train-validation F1 gap':best['Train-validation F1 gap'], 'Inner folds':len(fold['inner']),
        'Refit epochs':best['Refit epochs'], 'Final refit learning rate':learning_rates[-1]})
    display(pd.DataFrame(candidate_rows).drop(columns='Candidate index').round(4))
cnn_history = pd.concat(cnn_history_parts, ignore_index=True)
validation_results['CNN'] = pd.DataFrame(cnn_search_rows)
selection_summary = pd.DataFrame(selected_settings)
display(Markdown('**Selected settings for all models and outer splits**'))
display(selection_summary.round(4))


In [ ]:
for setting in ['Random','LOSO']:
    example_outer_fold = next(fold['name'] for fold in tuning_folds if fold['setting']==setting)
    choices = validation_results['CNN']
    choice = choices[(choices['Outer split']==example_outer_fold) & choices.Selected].iloc[0]
    curve = cnn_history[(cnn_history['Outer split']==example_outer_fold) &
                        (cnn_history.Candidate==choice['Candidate']) & (cnn_history['Inner fold']==1)]
    checkpoint_epoch = int(curve.loc[curve['Selected checkpoint'], 'Epoch'].iloc[0])
    fig, axes = plt.subplots(1,3,figsize=(15,3.5))
    axes[0].plot(curve.Epoch,curve['Training loss'],label='Training')
    axes[0].plot(curve.Epoch,curve['Validation loss'],label='Validation')
    axes[0].set(title='Evaluation-mode loss',xlabel='Epoch',ylabel='Cross-entropy loss')
    axes[0].axvline(checkpoint_epoch,color='#64748b',linestyle='--',label='Selected checkpoint')
    axes[0].legend()
    axes[1].plot(curve.Epoch,curve['Training macro F1'],label='Training')
    axes[1].plot(curve.Epoch,curve['Validation macro F1'],label='Validation')
    axes[1].set(title='Macro F1',xlabel='Epoch',ylabel='Macro F1',ylim=(0,1))
    axes[1].axvline(checkpoint_epoch,color='#64748b',linestyle='--')
    axes[1].legend()
    axes[2].step(curve.Epoch,curve['Learning rate'],where='post',color='#0d9488')
    axes[2].set(title='Learning rate',xlabel='Epoch',ylabel='Adam LR',yscale='log')
    fig.suptitle(f"{example_outer_fold}: {choice['Candidate']}, inner fold 1")
    fig.tight_layout()
    plt.show()

## 10. Final model prediction and evaluation

After all tuning is complete, predict the same outer test recordings used by the
baseline. Scikit-learn pipelines accept **raw** X and apply their fitted training
scaler; CNN bundles apply normalisation fitted on all outer training images.
DummyClassifier is reused unchanged. Optimised configurations can differ by outer
fold because each selection used only that fold's training data.

Report pooled out-of-fold accuracy and macro F1 for both protocols. Each
recording contributes once per protocol per stage. Also show the unweighted fold
mean ± sample standard deviation: five random scores and six LOSO scores.
The comparison tables include both stages. Accuracy changes are percentage points;
macro F1 changes are score differences. Negative changes are retained: tuning does
not guarantee improvement. The speaker-level table shows LOSO variation. Use inner
validation, not these outer-test results, to select settings for further experiments.

All classical pipelines use every feature in the selected basic/rich preset;
the input representation and outer test recordings remain fixed across stages.
Speaker-level tables report both accuracy and macro F1. Training/validation
diagnostics explain gaps but never choose an outer-test winner. Further designs
made after inspecting outer results remain exploratory until independently confirmed.

In [ ]:
if (FEATURE_SET != baseline_feature_set or X.shape[1] != baseline_feature_count
        or tuple(Z.shape[1:]) != baseline_cnn_shape):
    raise RuntimeError('Feature preset changed: re-run sections 4–9 before final evaluation.')
missing = [(fold['name'],name) for fold in tuning_folds for name in model_order
           if name not in optimised_models[fold['name']]]
if missing:
    raise RuntimeError(f'Run every tuning subsection first. Missing models: {missing}')
baseline_lookup = {(record['Fold'],record['Model']):record for record in baseline_prediction_records}
optimised_predictions,optimised_prediction_records,speaker_rows = [],[],[]
for fold in tuning_folds:
    test = fold['test']
    for name in model_order:
        fitted = optimised_models[fold['name']][name]
        if name=='CNN':
            model = fitted['model'].to(DEVICE)
            predicted = cnn_predict(model,(Z[test]-fitted['image_mean'])/fitted['image_std'])
            model.cpu()
        else:
            predicted = fitted.predict(X[test])
        actual = y[test]
        optimised_predictions.append((fold['setting'],name,actual.copy(),predicted.copy()))
        optimised_prediction_records.append({'Protocol':fold['setting'],'Fold':fold['name'],'Model':name,
            'Indices':test.copy(),'Actual':actual.copy(),'Predicted':predicted.copy()})
        baseline_record = baseline_lookup[(fold['name'],name)]
        np.testing.assert_array_equal(baseline_record['Indices'],test)
        np.testing.assert_array_equal(baseline_record['Actual'],actual)
        if fold['setting']=='LOSO':
            for stage,values in [('Baseline',baseline_record['Predicted']),('Optimised',predicted)]:
                speaker_rows.append({'Model':name,'Stage':stage,'Speaker':metadata.speaker.iloc[test].iloc[0],
                    'Accuracy':accuracy_score(actual,values),
                    'Macro F1':f1_score(actual,values,average='macro',labels=labels,zero_division=0)})
optimised_summary,optimised_fold_scores,optimised_fold_summary = summarise_predictions(optimised_prediction_records)
final_summary = pd.concat({'Baseline':baseline_summary,'Optimised':optimised_summary},names=['Stage','Model'])
final_summary = final_summary.reorder_levels(['Model','Stage']).reindex(
    pd.MultiIndex.from_product([model_order,['Baseline','Optimised']],names=['Model','Stage']))
display(final_summary.style.format(metric_format).set_caption(
    f'Pooled out-of-fold final comparison — {FEATURE_SET}, {X.shape[1]} acoustic features; CNN {CNN_INPUT_NAME}'))
final_fold_summary = pd.concat({'Baseline':baseline_fold_summary,'Optimised':optimised_fold_summary},
                               names=['Stage','Model','Protocol']).reorder_levels(['Model','Stage','Protocol'])
final_fold_summary = final_fold_summary.reindex(pd.MultiIndex.from_product(
    [model_order,['Baseline','Optimised'],['Random','LOSO']],names=['Model','Stage','Protocol']))
show_fold_statistics(final_fold_summary,'Baseline versus optimised fold mean and standard deviation')
final_fold_scores = pd.concat([baseline_fold_scores.assign(Stage='Baseline'),
                              optimised_fold_scores.assign(Stage='Optimised')],ignore_index=True)
improvement = optimised_summary-baseline_summary
improvement = improvement.rename(columns={'Random accuracy':'Random accuracy change (pp)',
    'Random macro F1':'Random macro F1 change', 'LOSO accuracy':'LOSO accuracy change (pp)',
    'LOSO macro F1':'LOSO macro F1 change'})
for column in ['Random accuracy change (pp)','LOSO accuracy change (pp)']:
    improvement[column] *= 100
display(improvement.style.format('{:+.3f}').set_caption('Optimised minus baseline'))
speaker_results = pd.DataFrame(speaker_rows)
display(speaker_results.pivot(index=['Model','Stage'],columns='Speaker',values='Accuracy')
        .style.format('{:.1%}').set_caption('LOSO accuracy for each held-out speaker'))
display(speaker_results.pivot(index=['Model','Stage'],columns='Speaker',values='Macro F1')
        .style.format('{:.3f}').set_caption('LOSO macro F1 for each held-out speaker'))

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(13,9),sharex='col')
for row, setting in enumerate(['Random','LOSO']):
    for column, metric in enumerate(['accuracy','macro F1']):
        key = f'{setting} {metric}'
        plot_values = pd.DataFrame({'Baseline':baseline_summary[key],
                                    'Optimised':optimised_summary[key]}).reindex(model_order)
        plot_values.plot.barh(ax=axes[row,column],color=['#94a3b8','#2563eb'])
        axes[row,column].set(title=f"{'Random 5-fold CV (pooled)' if setting=='Random' else 'LOSO (pooled)'}: {metric}",
                            xlabel=metric,ylabel='',xlim=(0,1))
        axes[row,column].invert_yaxis()
        axes[row,column].grid(axis='x',alpha=0.2)
        axes[row,column].set_axisbelow(True)
        if metric=='accuracy':
            axes[row,column].xaxis.set_major_formatter(PercentFormatter(1))
fig.suptitle('Baseline versus optimised models: pooled out-of-fold test predictions')
fig.tight_layout()
plt.show()